In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import patches as mpatches
import imageio

In [2]:
year = '2022'

In [5]:
pd.set_option('display.max_rows', 500)
plt.style.use('seaborn-v0_8-dark')

In [7]:
# data from grafana monitoring of lichess.org (websocket connections as proxy for players)
df = pd.read_csv(f'~/git/wclichess/{year}/wcdata{year}.csv')

In [8]:
# fixtures from https://fixturedownload.com/results/fifa-world-cup-2022
fixtures = pd.read_csv(f'{year}/fifa-world-cup-{year}-UTC.csv')

In [10]:
df = df.ffill() # cover missing points in grafana data
df['Time'] = pd.to_datetime(df.Time)
df['players'] = pd.to_numeric(df['players'].str.split().str[0]) # from 60.0 K string to 60.0 float

In [11]:
df['day'] = df.Time.dt.date
df['daytime'] = df.Time.dt.strftime('%H-%M')

In [12]:
days = df.day.unique()[4:] # world cup days
standard_day = df.day.unique()[1] # normal day example

In [13]:
# create table to see what matches were played when and by which teams
fixture_lookup = fixtures[['Home Team', 'Away Team']].stack().reset_index()\
    .merge(fixtures['Date'], how='inner', left_on='level_0', right_index=True)\
    .drop(columns=['level_0', 'level_1'])\
    .rename(columns={0: 'Country'})
fixture_lookup['Day'] = pd.to_datetime(fixture_lookup.Date, dayfirst=True).dt.date
fixture_lookup['Time'] = pd.to_datetime(fixture_lookup.Date).dt.strftime('%H-%M')

/tmp/ipykernel_281623/4108941204.py:7: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  fixture_lookup['Time'] = pd.to_datetime(fixture_lookup.Date).dt.strftime('%H-%M')


In [15]:
# some names were too long to label neatly
fixture_lookup['Country'] = fixture_lookup.Country.replace({'Korea Republic': 'Korea', 'Saudi Arabia': 'S Arabia'})

In [16]:
# getting the placement of the graph labels correct for each kickoff
kickoffs = {
    '10-00': ([40,40], [43, 105], 'k--'),
    '13-00': ([52,52], [43, 105], 'k--'),
    '15-00': ([60,60], [43, 105], 'k--'),
    '16-00': ([64,64], [43, 105], 'k--'),
    '19-00': ([76,76], [43, 105], 'k--'),
}
kickoff_labels = {
    '10-00': 0.42,
    '13-00': 0.52,
    '15-00': 0.57,
    '16-00': 0.60,
    '19-00': 0.70
}

In [18]:
# make all the graphs for each day with both normal day and the specified day
for n, day in enumerate(days):
    f, ax = plt.subplots(figsize=(10, 7))
    plot_df = df.query('day == @day')[['daytime', 'players']]
    plot_df.plot(x='daytime', y='players', ax=ax, legend=False, color='g')
    plot_df = df.query('day == @standard_day')[['daytime', 'players']]
    plot_df.plot(x='daytime', y='players', ax=ax, legend=False, color='black')
    plt.title('Lichess.org Users During Football World Cup Relative to Normal Day', fontsize='18')
    # add labels for each kickoff
    for t in kickoffs.keys():
        matches = fixture_lookup.query('Day == @days[@n] & Time == @t')
        if not matches.empty:
            plt.plot(*kickoffs[t], lw=1, dashes=[2,2])
            plt.figtext(kickoff_labels[t], 0.15, matches.Country.to_string(index=False), fontsize='medium')
    plt.figtext(0.15, 0.7, f'Date: {day}', fontsize='xx-large')
    # credit
    plt.figtext(0.15, 0.15, 'github @michael1241', fontsize='small')
    plt.xlabel('Time (UTC)', fontsize='15')
    plt.ylabel('Online Users in 1000s', fontsize='15')
    plt.ylim([30,110])
    normal_day_patch = mpatches.Patch(color='black', label='Normal traffic')
    wc_day_patch = mpatches.Patch(color='green', label='World Cup traffic')
    plt.legend(handles=[normal_day_patch, wc_day_patch], loc=0, prop={'size': 15})
    plt.savefig(f'./{year}/img/img_{n}.png', 
                transparent = False
               )
    plt.close()

In [12]:
frames = []
for n, day in enumerate(days):
    image = imageio.v2.imread(f'./img/img_{n}.png')
    frames.append(image)
# mkdir ./img/ first
imageio.mimsave('./img/output.gif', frames, fps = 0.4)